In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.curated_sales AS
SELECT
    f.order_number,
    f.line_item,

    f.order_date,
    f.year,
    f.month,

    f.customerkey,
    c.gender,
    c.continent,

    f.storekey,
    COALESCE(s.store_country, 'Online') AS country,

    f.product_key,
    p.category,
    p.subcategory,

    f.quantity,

    f.unit_price_usd,
    f.exchange,

    f.revenue_usd,
    f.delivery_days,

    f.channel

FROM electronics_cat.gold.fact_sales f
LEFT JOIN electronics_cat.gold.dim_customer c
    ON f.customerkey = c.customerkey
LEFT JOIN electronics_cat.gold.dim_product p
    ON f.product_key = p.productkey
LEFT JOIN electronics_cat.gold.dim_store s
    ON f.storekey = s.store_key;

In [0]:
SELECT * FROM electronics_cat.gold.curated_sales;

In [0]:
SELECT
    year,
    month,
    ROUND(SUM(revenue_usd),2) AS revenue_usd
FROM electronics_cat.gold.curated_sales
GROUP BY year, month
ORDER BY year, month;

In [0]:
WITH monthly AS (
    SELECT month, SUM(revenue_usd) AS revenue
    FROM electronics_cat.gold.curated_sales
    GROUP BY month
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM monthly
)
SELECT
    month,
    ROUND(revenue,2),
    ROUND(revenue * 100 / total_rev,2) AS percent
FROM monthly, total
ORDER BY revenue DESC
LIMIT 3;

In [0]:
SELECT
    year,
    month,
    ROUND(SUM(revenue_usd),2) AS revenue_usd
FROM electronics_cat.gold.curated_sales
WHERE year = 2021
GROUP BY year, month
ORDER BY month;

In [0]:
WITH monthly AS (
    SELECT month, SUM(revenue_usd) AS revenue
    FROM electronics_cat.gold.curated_sales
    WHERE year = 2021
    GROUP BY month
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM monthly
)
SELECT
    month,
    ROUND(revenue,2) AS revenue_usd,
    ROUND(revenue * 100 / total_rev,2) AS percent_of_total
FROM monthly, total
ORDER BY revenue DESC
LIMIT 3;

In [0]:
WITH peak_months AS (
    SELECT month
    FROM (
        SELECT month, SUM(revenue_usd) AS rev
        FROM electronics_cat.gold.curated_sales
        WHERE year = 2021
        GROUP BY month
        ORDER BY rev DESC
        LIMIT 3
    )
)
SELECT
    category,
    ROUND(SUM(revenue_usd),2) AS peak_month_revenue,
    ROUND(
        SUM(revenue_usd)*100 / SUM(SUM(revenue_usd)) OVER(),2
    ) AS percent_of_peak_total
FROM electronics_cat.gold.curated_sales
WHERE year = 2021
  AND month IN (SELECT month FROM peak_months)
GROUP BY category
ORDER BY peak_month_revenue DESC
LIMIT 3;

In [0]:
SELECT
    ROUND(AVG(delivery_days),2) AS avg_days,
    COUNT(*) AS total_orders
FROM electronics_cat.gold.curated_sales
WHERE delivery_days IS NOT NULL;

In [0]:
SELECT
    country,
    ROUND(AVG(delivery_days),2) AS avg_days,
    COUNT(*) AS order_count,
    PERCENTILE(delivery_days,0.5) AS median_days
FROM electronics_cat.gold.curated_sales
WHERE delivery_days IS NOT NULL
GROUP BY country
ORDER BY avg_days DESC
LIMIT 5;

In [0]:
SELECT
    continent,

    ROUND(
        SUM(CASE WHEN channel='online' THEN revenue_usd END) /
        NULLIF(COUNT(DISTINCT CASE WHEN channel='online' THEN order_number END),0)
    ,2) AS aov_online,

    ROUND(
        SUM(CASE WHEN channel='store' THEN revenue_usd END) /
        NULLIF(COUNT(DISTINCT CASE WHEN channel='store' THEN order_number END),0)
    ,2) AS aov_store,

    COUNT(DISTINCT CASE WHEN channel='online' THEN order_number END) AS online_orders,
    COUNT(DISTINCT CASE WHEN channel='store' THEN order_number END) AS store_orders

FROM electronics_cat.gold.curated_sales
GROUP BY continent;

In [0]:
SELECT
    ROW_NUMBER() OVER(ORDER BY SUM(quantity) DESC) AS rank,
    category,
    SUM(quantity) AS units_sold,
    ROUND(SUM(quantity)*100 / SUM(SUM(quantity)) OVER(),2) AS percent
FROM electronics_cat.gold.curated_sales
GROUP BY category
LIMIT 5;

In [0]:
SELECT
    ROW_NUMBER() OVER(ORDER BY SUM(revenue_usd) DESC) AS rank,
    category,
    ROUND(SUM(revenue_usd),2) AS revenue,
    ROUND(SUM(revenue_usd)*100 / SUM(SUM(revenue_usd)) OVER(),2) AS percent
FROM electronics_cat.gold.curated_sales
GROUP BY category
LIMIT 5;

In [0]:
SELECT
    continent,
    gender,
    COUNT(DISTINCT customerkey) AS customer_count,
    ROUND(SUM(revenue_usd),2) AS total_spend,
    ROUND(SUM(revenue_usd)/COUNT(DISTINCT customerkey),2) AS avg_spend
FROM electronics_cat.gold.curated_sales
GROUP BY continent, gender;

In [0]:
WITH cust_orders AS (
    SELECT
        continent,
        customerkey,
        COUNT(DISTINCT order_number) AS orders
    FROM electronics_cat.gold.curated_sales
    GROUP BY continent, customerkey
)
SELECT
    continent,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN orders >=2 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(
        SUM(CASE WHEN orders >=2 THEN 1 ELSE 0 END)*100/COUNT(*),2
    ) AS repeat_rate
FROM cust_orders
GROUP BY continent;